# Examining gene and feature importance scores with and without germline data
- germline
- somatic
- somatic plus germline

Positive control to verify: Do we see BRCA2 in top genes for germline data? If not, suggests that we need to change how we manipulate the germline data (e.g. patho variant filtering, gene filtering, grouping variants by type, etc).

In [8]:
import logging
import os

import pandas as pd

from pnet import prostate_data_loaders

logging.basicConfig(
    format="%(asctime)s %(levelname)-8s %(message)s",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
)

logger = logging.getLogger()
logger.setLevel(logging.INFO)

%load_ext autoreload
%autoreload 2

In [ ]:
logging.debug("Defining paths for germline data")
GERMLINE_DATADIR = "../../pnet_germline/data/"
logging.debug("Defining paths for the sample metadata")
id_map_f = os.path.join(
    GERMLINE_DATADIR, "prostate/germline_somatic_id_map_outer_join.csv"
)  # germline_somatic_id_map_f
sample_metadata_f = os.path.join(
    GERMLINE_DATADIR,
    "prostate/pathogenic_variants_with_clinical_annotation_1341_aug2021_correlation.csv",
)

prostate_response = prostate_data_loaders.get_target(
    id_map_f, sample_metadata_f, id_to_use="Tumor_Sample_Barcode", target_col="is_met"
)
prostate_response = prostate_response.rename(columns={"is_met": "response"})


2025-04-11 17:58:24 INFO     Getting prediction target
2025-04-11 17:58:24 INFO     Loading the sample metadata DF that has all the IDs and also our target, metastatic status ('is_met')
2025-04-11 17:58:24 INFO     Loading the germline metadata file at ../../pnet_germline/data/prostate/pathogenic_variants_with_clinical_annotation_1341_aug2021_correlation.csv
2025-04-11 17:58:24 INFO     Generating the target DF (target column '{target_col}' indexed by '{id}')
2025-04-11 17:58:24 INFO     Target column value_counts: 0    704
1    626
Name: is_met, dtype: int64


# Germline experiments
How does adding germline data affect the top N feature importances, rank ordering? 
- What new genes do we see? 
- How do magnitudes change? 
- How do ranks change?

# BDT gene importances: germline data experiment
- germline
- somatic
- somatic + germline

In [ ]:
logging.info("Directories from before I ran variant-level QC.")
MODEL_TYPE = "bdt"
EVALUATION_SET = "validation"  #'test' # val

dirs = [
    f"{MODEL_TYPE}_eval_set_{EVALUATION_SET}_germline",
    f"{MODEL_TYPE}_eval_set_{EVALUATION_SET}_somatic",
    f"{MODEL_TYPE}_eval_set_{EVALUATION_SET}_somatic_and_germline",
]
dirs

2025-04-11 18:02:57 INFO     Directories from before I ran variant-level QC.


['bdt_eval_set_validation_germline',
 'bdt_eval_set_validation_somatic',
 'bdt_eval_set_validation_somatic_and_germline']

In [ ]:
dirs_dict = {
    f"{MODEL_TYPE}_eval_set_{EVALUATION_SET}_germline": f"{MODEL_TYPE}_eval_set_{EVALUATION_SET}_variantQCed_wandbID_s6b9gk6n",
    f"{MODEL_TYPE}_eval_set_{EVALUATION_SET}_somatic": f"{MODEL_TYPE}_eval_set_{EVALUATION_SET}_variantQCed_wandbID_ctutnvck",
    f"{MODEL_TYPE}_eval_set_{EVALUATION_SET}_somatic_and_germline": f"{MODEL_TYPE}_eval_set_{EVALUATION_SET}_variantQCed_wandbID_muho5uft",
}

dirs = list(dirs_dict.values())

In [ ]:
# Ran this once to change the format of the saved-down lists
# for i in dirs:
#     imps = pd.read_csv(os.path.join(i, f'{EVALUATION_SET}_gene_feature_importances.csv'.format(i))).set_index('Unnamed: 0')
#     imps = imps.reset_index()
#     imps.columns.name = None
#     imps.columns = ['feature', 'importance score']
#     imps.to_csv(os.path.join(i, f'{EVALUATION_SET}_gene_feature_importances.csv'.format(i)), index=False)
#     display(imps)


In [ ]:
bdt_gene_imps = []
for i in dirs:
    base = "../../pnet/results"
    imps = pd.read_csv(
        os.path.join(base, i, f"{EVALUATION_SET}_gene_feature_importances.csv")
    ).set_index("feature")
    imps.columns.name = None
    bdt_gene_imps.append(imps)

    ranks = (
        imps.rank(ascending=False, method="dense")
        .astype(int)
        .sort_values(by="importance score", ascending=True)
    )
    display(ranks)
    logging.debug(sorted(ranks["importance score"].tolist()))

,importance score
feature,
PCA1,1
PCA8,2
PCA2,3
PCA10,4
PCA6,5
...,...
BRCA1_germline_mut,45
EWSR1_germline_mut,45
TOP1_germline_mut,45


,importance score
feature,
AR_somatic_amp,1
AR_somatic_mut,2
TP53_somatic_mut,3
PCA8,4
PCA2,5
...,...
BIRC6_somatic_del,131
FANCD2_somatic_del,131
COL3A1_somatic_del,131


,importance score
feature,
AR_somatic_amp,1
AR_somatic_mut,2
TP53_somatic_mut,3
PCA8,4
PCA2,5
...,...
PMS1_somatic_del,127
ROS1_somatic_del,127
VTI1A_somatic_del,127


In [ ]:
## Here we create a dataset x feature rank DF (values = feature name). This is useful for comparing across dataset configurations. The input was a list of series, where each series is the gene imp list from a given run.
N = 25

# Create a DataFrame to store the ranks
rank_df = pd.DataFrame()
tmp_ranks = []
series_list = bdt_gene_imps
series_list_names = dirs  # optional
# series_list = rf_gene_imps
# Iterate through each series, calculate ranks, and add to the DataFrame
for i in range(len(series_list)):
    series = series_list[i]
    # Calculate ranks and convert them to integers
    ranks = (
        series.abs()
        .rank(ascending=False, method="dense")
        .astype(int)
        .sort_values(by="importance score")
    )
    tmp_ranks.append(ranks.index.tolist())
    # Add ranks to the DataFrame
    # rank_df[series.name] = ranks

# Display the resulting DataFrame
rank_df = pd.DataFrame(tmp_ranks)
rank_df.index = series_list_names

rank_df.index = list(dirs_dict.keys())  # TODO: this is a renaming based on dirs_dict

display(rank_df.loc[:, :N].T)


,bdt_eval_set_validation_germline,bdt_eval_set_validation_somatic,bdt_eval_set_validation_somatic_and_germline
0,PCA1,AR_somatic_amp,AR_somatic_amp
1,PCA8,AR_somatic_mut,AR_somatic_mut
2,PCA2,TP53_somatic_mut,TP53_somatic_mut
3,PCA10,PCA8,PCA8
4,PCA6,PCA2,PCA2
5,BRCA2_germline_mut,PCA1,PCA10
6,PCA9,PCA10,PCA1
7,PCA5,PCA7,PCA7
8,WNK2_germline_mut,PCA3,STRN_somatic_amp
9,PCA7,STRN_somatic_amp,BRCA2_germline_mut


In [ ]:
print(rank_df.loc[:, :N].stack().value_counts())
top_unique_features = rank_df.loc[:, :N].stack().unique()
print([i.split("_")[0] for i in top_unique_features])
print(len(top_unique_features))


# Display a gene x rank DF (value = # times that gene had that rank across the N=20 runs)
top_gene_by_rank_consistency_df = (
    rank_df.loc[:, :N].apply(lambda col: col.value_counts()).fillna("").reindex(top_unique_features)
)
display(top_gene_by_rank_consistency_df)

PCA1                   3
PCA2                   3
PCA10                  3
PCA6                   3
PCA7                   3
PCA4                   3
PCA3                   3
PCA8                   3
NOTCH1_somatic_mut     2
AR_somatic_amp         2
STRN_somatic_amp       2
PRF1_somatic_amp       2
EPCAM_somatic_amp      2
MUC4_somatic_mut       2
SAMD9L_somatic_mut     2
ZEB1_somatic_del       2
MLH1_somatic_del       2
PMS2_somatic_amp       2
APC_somatic_del        2
TP53_somatic_mut       2
FCRL4_somatic_amp      2
WNK2_germline_mut      2
PCA5                   2
FAT1_somatic_mut       2
BRCA2_germline_mut     2
AR_somatic_mut         2
TOP1_somatic_del       1
PTCH2_somatic_amp      1
PCM1_germline_mut      1
FCRL4_germline_mut     1
MITF_germline_mut      1
REL_germline_mut       1
CYP2C8_germline_mut    1
PSIP1_germline_mut     1
ECT2L_germline_mut     1
NT5C2_germline_mut     1
FANCD2_germline_mut    1
WRN_germline_mut       1
POLQ_germline_mut      1
ERCC2_germline_mut     1


,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
PCA1,1.0,,,,,1.0,1.0,,,,...,,,,,,,,,,
PCA8,,1.0,,2.0,,,,,,,...,,,,,,,,,,
PCA2,,,1.0,,2.0,,,,,,...,,,,,,,,,,
PCA10,,,,1.0,,1.0,1.0,,,,...,,,,,,,,,,
PCA6,,,,,1.0,,,,,,...,,,,,1.0,,,,,
BRCA2_germline_mut,,,,,,1.0,,,,1.0,...,,,,,,,,,,
PCA9,,,,,,,1.0,,,,...,,,,,,,,,,
PCA5,,,,,,,,1.0,,,...,,,,,,,,,,
WNK2_germline_mut,,,,,,,,,1.0,,...,,,,,1.0,,,,,
PCA7,,,,,,,,2.0,,1.0,...,,,,,,,,,,


### Sorting ranks for each dataset configuration

In [ ]:
# bdt_gene_imps = stability_utils.get_sklearn_feature_imps(SAVEDIR)
# bdt_model_stability = stability_utils.calc_model_stability(bdt_gene_imps, n_top_genes=50)
# bdt_model_stability

# ## Here we create a runs x feature rank DF (values = feature name). This is useful for comparing across runs. The input was a list of series, where each series is the gene imp list from a given run.
# # Create a DataFrame to store the ranks
# rank_df = pd.DataFrame()
# tmp_ranks = []
# series_list = bdt_gene_imps
# # series_list = rf_gene_imps
# # Iterate through each series, calculate ranks, and add to the DataFrame
# for i in range(len(series_list)):
#     series = series_list[i]
#     # Calculate ranks and convert them to integers
#     ranks = series.abs().rank(ascending=False, method='dense').astype(int).sort_values()
#     tmp_ranks.append(ranks.index.tolist())
#     # Add ranks to the DataFrame
#     # rank_df[series.name] = ranks

# # Display the resulting DataFrame
# N = 10
# rank_df = pd.DataFrame(tmp_ranks)
# display(rank_df.loc[:,:N])

# print(rank_df.loc[:,:N].stack().value_counts())
# top_unique_features = rank_df.loc[:,:N].stack().unique()
# print([i.split("_")[0] for i in top_unique_features])
# print(len(top_unique_features))


# # Display a gene x rank DF (value = # times that gene had that rank across the N=20 runs)
# top_gene_by_rank_consistency_df = rank_df.loc[:,:N].apply(lambda col: col.value_counts()).fillna('').reindex(top_unique_features)
# display(top_gene_by_rank_consistency_df)

# P-NET gene importances: germline data experiment 
- germline
- somatic
- somatic + germline
## (averaged across N=20 runs)

In [ ]:
who = "val"
dirs = ["prostate_val_germline", "prostate_val_somatic", "prostate_val_germ_and_somatic"]

In [ ]:
df_imps = pd.DataFrame()
df_ranks = pd.DataFrame()
for i in dirs:
    imps = pd.read_csv(f"../results/{i}/{who}_gene_importances.csv".format(i)).set_index(
        "Unnamed: 0"
    )
    imps = imps.join(prostate_response).groupby("response").mean().diff(axis=0).iloc[1]
    ranks = imps.rank(ascending=False)
    df_imps[i] = imps
    df_ranks[i] = ranks

logging.info("Averaged across trials, top importance genes")
df_imps.mean(axis=1).nlargest(20)


2025-04-11 18:07:07 INFO     Averaged across trials, top importance genes


AR         0.155441
DDX21      0.095672
TP53       0.068426
PRSS1      0.040375
MUC4       0.024228
BVES       0.021105
AMZ1       0.019490
OBSCN      0.017219
RAC1       0.017088
AVPR1B     0.015395
AP2A2      0.014658
COL11A1    0.013686
NUP133     0.013019
MAN1C1     0.012853
ACVR2A     0.012706
HAX1       0.012608
RB1        0.012361
WWP1       0.011975
FCRL1      0.011437
AFP        0.011409
dtype: float64

In [ ]:
for i in dirs:
    logging.info(f"Sorting by {i}")
    df_imps = df_imps.sort_values(by=i, ascending=False)
    df_ranks = df_ranks.sort_values(by=i, ascending=True)
    display(df_imps[:10])
    # display(df_ranks[:10])

2025-04-11 18:07:15 INFO     Sorting by prostate_val_germline


,prostate_val_germline,prostate_val_somatic,prostate_val_germ_and_somatic
BRCA2,0.002892,0.015489,2.465071e-03
ABL2,0.001204,-0.000105,-1.002356e-03
HFE,0.000935,0.006085,4.897047e-03
HLA-A,0.000542,0.000000,2.074946e-02
FBLN2,0.000265,0.000000,5.257273e-05
REL,0.000211,0.005553,2.677447e-05
EML4,0.000200,0.000000,-1.542780e-10
FANCD2,0.000166,0.000242,-1.593299e-04
WNK2,0.000082,0.000139,2.737360e-07
HNF1A,0.000045,0.000377,-2.181008e-03


2025-04-11 18:07:15 INFO     Sorting by prostate_val_somatic


,prostate_val_germline,prostate_val_somatic,prostate_val_germ_and_somatic
DDX21,0.0,0.285545,1.469577e-03
BVES,0.0,0.063316,-3.685508e-07
AMZ1,0.0,0.058493,-2.439446e-05
COL11A1,0.0,0.042417,-1.359398e-03
AVPR1B,0.0,0.039716,6.468787e-03
ACVR2A,0.0,0.037865,2.540383e-04
HAX1,0.0,0.037829,-6.620649e-06
ITGA1,0.0,0.035291,-2.885075e-02
FCRL1,0.0,0.034322,-1.046204e-05
SPACA1,0.0,0.034043,2.003635e-05


2025-04-11 18:07:15 INFO     Sorting by prostate_val_germ_and_somatic


,prostate_val_germline,prostate_val_somatic,prostate_val_germ_and_somatic
AR,0.000000,-0.047968,0.514290
TP53,0.000000,0.031098,0.174181
PRSS1,-0.001785,-0.001467,0.124376
MUC4,0.000000,0.000500,0.072185
RAC1,0.000000,0.002075,0.049191
RB1,0.000000,-0.002194,0.039277
GNAS,0.000000,-0.033860,0.039031
AP2A2,0.000000,0.006237,0.037736
OBSCN,0.000000,0.015571,0.036086
PDGFA,0.000000,-0.002520,0.033259
